# Email Spam Classification

Classify emails as spam or not spam using KNN and SVM, and compare their performance.

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
balaka18_email_spam_classification_dataset_csv_path = kagglehub.dataset_download('balaka18/email-spam-classification-dataset-csv')
balaka18_email_spam_classification_path = kagglehub.notebook_output_download('balaka18/email-spam-classification')
print('Data source import complete.')

Using Colab cache for faster access to the 'email-spam-classification-dataset-csv' dataset.
Extracting files...
Data source import complete.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/email-spam-classification-dataset-csv/emails.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

df = pd.read_csv("/kaggle/input/email-spam-classification-dataset-csv/emails.csv")
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
print(df.head())

Dataset shape: (5172, 3002)

First five rows:
  Email No.  the  to  ect  and  for  of    a  you  hou  ...  connevey  jay  \
0   Email 1    0   0    1    0    0   0    2    0    0  ...         0    0   
1   Email 2    8  13   24    6    6   2  102    1   27  ...         0    0   
2   Email 3    0   0    1    0    0   0    8    0    0  ...         0    0   
3   Email 4    0   5   22    0    5   1   51    2   10  ...         0    0   
4   Email 5    7   6   17    1    5   2   57    0    9  ...         0    0   
   valued  lay  infrastructure  military  allowing  ff  dry  Prediction  
0       0    0               0         0         0   0    0           0  
1       0    0               0         0         0   1    0           0  
2       0    0               0         0         0   0    0           0  
3       0    0               0         0         0   0    0           0  
4       0    0               0         0         0   1    0           0  
[5 rows x 3002 columns]


In [ ]:
df = df.drop_duplicates()
target_column = df.columns[-1]
id_columns = [
    col for col in df.columns
    if col.strip().lower() in ["email no.", "email no", "email", "email_name", "email name"]
]
X = df.drop(columns=[target_column] + id_columns, errors="ignore")
y = df[target_column]
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)
y = pd.to_numeric(y, errors="coerce")
valid_rows = y.notna() & y.isin([0, 1])
X = X.loc[valid_rows]
y = y.loc[valid_rows].astype(int)

print("\nFeatures shape:", X.shape)
print("Target distribution (0 = Not Spam, 1 = Spam):")
print(y.value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)

Features shape: (5172, 2999)
Target distribution (0 = Not Spam, 1 = Spam):
Prediction
0    3672
1    1500
Name: count, dtype: int64


In [ ]:
svm = SVC(kernel="linear")
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)

def evaluate_model(name, y_true, y_pred):
    print(f"\n{'=' * 55}\n{name}\n{'=' * 55}")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["Not Spam", "Spam"],
        zero_division=0
    ))

evaluate_model("K-Nearest Neighbors (KNN)", y_test, knn_pred)
evaluate_model("Support Vector Machine (SVM)", y_test, svm_pred)

K-Nearest Neighbors (KNN)
Accuracy : 0.8291
Precision: 0.6380
Recall   : 0.9493
F1-score : 0.7631

Confusion Matrix:
[[716 202]
 [ 19 356]]

Classification Report:
              precision    recall  f1-score   support
    Not Spam       0.97      0.78      0.87       918
        Spam       0.64      0.95      0.76       375
    accuracy                           0.83      1293
   macro avg       0.81      0.86      0.81      1293
weighted avg       0.88      0.83      0.84      1293
Support Vector Machine (SVM)
Accuracy : 0.9497
Precision: 0.9189
Recall   : 0.9067
F1-score : 0.9128

Confusion Matrix:
[[888  30]
 [ 35 340]]

Classification Report:
              precision    recall  f1-score   support
    Not Spam       0.96      0.97      0.96       918
        Spam       0.92      0.91      0.91       375
    accuracy                           0.95      1293
   macro avg       0.94      0.94      0.94      1293
weighted avg       0.95      0.95      0.95      1293


In [ ]:
results = pd.DataFrame({
    "Model": ["KNN (k=5)", "SVM (linear kernel)"],
    "Accuracy": [
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, svm_pred)
    ],
    "Precision": [
        precision_score(y_test, knn_pred, zero_division=0),
        precision_score(y_test, svm_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, knn_pred, zero_division=0),
        recall_score(y_test, svm_pred, zero_division=0)
    ],
    "F1-score": [
        f1_score(y_test, knn_pred, zero_division=0),
        f1_score(y_test, svm_pred, zero_division=0)
    ]
})
print("\nModel comparison:")
print(results.to_string(index=False, formatters={
    "Accuracy": "{:.4f}".format,
    "Precision": "{:.4f}".format,
    "Recall": "{:.4f}".format,
    "F1-score": "{:.4f}".format
}))

Model comparison:
              Model Accuracy Precision Recall F1-score
          KNN (k=5)   0.8291    0.6380 0.9493   0.7631
SVM (linear kernel)   0.9497    0.9189 0.9067   0.9128
